# Bank XYZ — AI Analyst Integration
**Tujuan:** Test & konfigurasi Claude API untuk fitur AI di dashboard

**Penting:** Data bersifat confidential — nama bank harus dianonim sebagai 'Bank XYZ'

---
### Struktur Notebook
1. Setup & konfigurasi API key
2. Build data context (ringkasan data untuk dikirim ke AI)
3. Test Executive Summary generation
4. Test Q&A Analyst
5. Test Auto Narrative (untuk presentasi)
6. Finalisasi system prompt untuk dashboard

In [1]:
# Install package Groq (jalankan sekali)
import subprocess
subprocess.run(["pip", "install", "groq", "--quiet"], check=True)
print("✓ groq terinstall")

✓ groq terinstall


In [2]:
import os
import json
import pandas as pd
import numpy as np
from groq import Groq

# ── API Configuration (Groq) ───────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\.env')
except ImportError:
    pass

API_KEY = os.getenv('GROQ_API_KEY')
MODEL   = 'llama-3.3-70b-versatile'  # Model Groq gratis, kualitas tinggi

if not API_KEY:
    print('⚠️  GROQ_API_KEY belum di-set!')
    print('   Buka .env dan tambahkan: GROQ_API_KEY=gsk_...')
else:
    client = Groq(api_key=API_KEY)
    print(f'✓ Groq API siap: {API_KEY[:12]}...')
    print(f'  Model: {MODEL}')

✓ Groq API siap: gsk_7VJcHHa0...
  Model: llama-3.3-70b-versatile


## 1. Build Data Context

In [3]:
# ── Load semua data ────────────────────────────────────────────
DATA_DIR = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data'

def nps_score(series):
    s = series.dropna()
    if len(s) == 0: return np.nan
    return float(round(((s >= 9).sum() - (s <= 6).sum()) / len(s) * 100, 1))

master    = pd.read_csv(f'{DATA_DIR}/processed_bankxyz.csv')
branch    = pd.read_csv(f'{DATA_DIR}/agg_branch.csv')
prov      = pd.read_csv(f'{DATA_DIR}/agg_provinsi.csv')
ipa       = pd.read_csv(f'{DATA_DIR}/ipa_matrix.csv')
emo       = pd.read_csv(f'{DATA_DIR}/emotion_summary.csv')
comp      = pd.read_csv(f'{DATA_DIR}/competitor_benchmark.csv')
nps_comp  = pd.read_csv(f'{DATA_DIR}/nps_competitor.csv')
brand     = pd.read_csv(os.path.join(DATA_DIR, 'brand_perception.csv'))
driver    = pd.read_csv(f'{DATA_DIR}/driver_analysis.csv') if os.path.exists(f'{DATA_DIR}/driver_analysis.csv') else None

# ── Ringkas data untuk context ────────────────────────────────
gkpi = {
    'nps': nps_score(master['nps_num']),
    'csi': round(float(master['csi_num'].mean()), 2),
    'loyalty': round(float(master['loyalty_num'].mean()), 2),
    'total': len(master),
    'provinces': master['provinsi'].nunique(),
    'branches': master['cabang'].nunique(),
    'promoters': int((master['nps_num'] >= 9).sum()),
    'detractors': int((master['nps_num'] <= 6).sum()),
}

top3_branch   = branch.nlargest(3, 'nps_score')[['CABANG','PROV','nps_score']].to_dict('records')
bot3_branch   = branch.nsmallest(3, 'nps_score')[['CABANG','PROV','nps_score']].to_dict('records')
quick_wins    = ipa[ipa['kuadran']=='Quick Win'].nlargest(5,'importance')[['kategori','atribut_idx','importance','performance']].to_dict('records')
top_emo_pos   = emo[emo['tipe']=='positif'].nlargest(3,'mean_score')[['emosi','mean_score','pct_strong']].to_dict('records')
top_emo_neg   = emo[emo['tipe']=='negatif'].nlargest(3,'mean_score')[['emosi','mean_score','pct_strong']].to_dict('records')
brand_gap     = brand.nlargest(3,'selisih')[['atribut','xyz_pct_agree','komp_pct_agree','selisih']].to_dict('records')

def build_context(filters=None):
    """Build context string untuk dikirim ke Claude API."""
    ctx = (
        f"DATA SURVEI KEPUASAN NASABAH — BANK XYZ (CONFIDENTIAL)\n"
        f"Catatan: Nama bank telah dianonimkan. Jangan sebut nama bank asli.\n\n"
        f"=== KPI UTAMA ===\n"
        f"- Total responden: {gkpi['total']:,} dari {gkpi['provinces']} provinsi, {gkpi['branches']} cabang\n"
        f"- NPS Score: {gkpi['nps']} (Promoter {round(gkpi['promoters']/gkpi['total']*100,1)}%, Detractor {round(gkpi['detractors']/gkpi['total']*100,1)}%)\n"
        f"- CSI Mean: {gkpi['csi']}/6 ({round((master['csi_num']>=5).mean()*100,1)}% Sangat Puas)\n"
        f"- Loyalty Mean: {gkpi['loyalty']}/6\n"
        f"- NPS vs Kompetitor: Bank XYZ {nps_comp.iloc[0]['nps_score']} vs Kompetitor {nps_comp.iloc[1]['nps_score'] if len(nps_comp)>1 else 'N/A'}\n\n"
        f"=== CABANG TERBAIK ===\n"
        + '\n'.join([f"- {b['CABANG']} ({b['PROV']}): NPS {b['nps_score']}" for b in top3_branch])
        + f"\n\n=== CABANG PERLU PERHATIAN ===\n"
        + '\n'.join([f"- {b['CABANG']} ({b['PROV']}): NPS {b['nps_score']}" for b in bot3_branch])
        + f"\n\n=== QUICK WIN TOUCHPOINTS (High Importance, Low Performance) ===\n"
        + '\n'.join([f"- [{qw['kategori']}] {str(qw['atribut_idx'])[:50]} (Imp: {qw['importance']}, Perf: {qw['performance']})" for qw in quick_wins])
        + f"\n\n=== EMOSI NASABAH ===\n"
        + f"Positif tertinggi: " + ', '.join([f"{e['emosi']} ({e['pct_strong']}%)" for e in top_emo_pos])
        + f"\nNegatif tertinggi: " + ', '.join([f"{e['emosi']} ({e['pct_strong']}%)" for e in top_emo_neg])
        + f"\n\n=== KEUNGGULAN BRAND vs KOMPETITOR (% setuju) ===\n"
        + '\n'.join([f"- {b['atribut'][:50]}: XYZ {b['xyz_pct_agree']}% vs Komp {b['komp_pct_agree']}% (selisih +{b['selisih']}%)" for b in brand_gap])
    )

    if driver is not None:
        top_driver = driver.head(3)
        driver_lines = '\n'.join([f"- {row['touchpoint']}: r={row['correlation']}" for _, row in top_driver.iterrows()])
        ctx += "\n\n=== DRIVER NPS TERTINGGI ===\n" + driver_lines

    return ctx

CONTEXT = build_context()
print(CONTEXT)


DATA SURVEI KEPUASAN NASABAH — BANK XYZ (CONFIDENTIAL)
Catatan: Nama bank telah dianonimkan. Jangan sebut nama bank asli.

=== KPI UTAMA ===
- Total responden: 1,730 dari 14 provinsi, 128 cabang
- NPS Score: 80.9 (Promoter 82.3%, Detractor 1.4%)
- CSI Mean: 5.89/6 (99.6% Sangat Puas)
- Loyalty Mean: 5.87/6
- NPS vs Kompetitor: Bank XYZ 48.0 vs Kompetitor 9.8

=== CABANG TERBAIK ===
- Denpasar 2 (Bali): NPS 100.0
- Lebak 1 (Banten): NPS 100.0
- Pandeglang 3 (Banten): NPS 100.0

=== CABANG PERLU PERHATIAN ===
- Bogor 1 (Jawa Barat): NPS -22.2
- Semarang 2 (Jawa Tengah): NPS 0.0
- Tangerang 3 (Banten): NPS 11.1

=== QUICK WIN TOUCHPOINTS (High Importance, Low Performance) ===


=== EMOSI NASABAH ===
Positif tertinggi: Aman (99.0%), Percaya (98.8%), Dihargai (98.6%)
Negatif tertinggi: Tergesa-gesa (10.4%), Tertekan (10.9%), Diabaikan (10.7%)

=== KEUNGGULAN BRAND vs KOMPETITOR (% setuju) ===
- Percaya diri bertransaksi: XYZ 98.6% vs Komp 94.0% (selisih +4.6%)
- Banyak ATM: XYZ 99.9% vs Kom

In [4]:
print(brand.columns.tolist())
print(brand.head(2))

['atribut', 'xyz_pct_agree', 'komp_pct_agree', 'selisih']
                  atribut  xyz_pct_agree  komp_pct_agree  selisih
0           Bank terkenal           99.9            97.1      2.8
1  Digunakan banyak orang           99.8            97.8      2.0


## 2. System Prompt Design

In [5]:
# ── System prompt untuk dashboard ────────────────────────────
SYSTEM_PROMPT = """Kamu adalah AI Analyst profesional yang membantu tim analitik menginterpretasi 
hasil survei kepuasan nasabah Bank XYZ.

KETENTUAN PENTING:
1. Data bersifat CONFIDENTIAL — selalu sebut 'Bank XYZ', JANGAN sebut nama bank asli
2. Jawab dalam Bahasa Indonesia yang profesional dan mudah dipahami manajer bank
3. Selalu sertakan angka/data spesifik dari konteks yang diberikan
4. Fokus pada insight yang ACTIONABLE — apa yang harus dilakukan, bukan hanya apa yang terjadi
5. Maksimal 3 paragraf per jawaban, kecuali diminta narasi panjang
6. Gunakan bullet point untuk rekomendasi bila perlu

FORMAT JAWABAN:
- Mulai dengan insight utama (1 kalimat)
- Elaborasi dengan data pendukung
- Akhiri dengan rekomendasi konkret"""

print('System prompt:')
print(SYSTEM_PROMPT)

System prompt:
Kamu adalah AI Analyst profesional yang membantu tim analitik menginterpretasi 
hasil survei kepuasan nasabah Bank XYZ.

KETENTUAN PENTING:
1. Data bersifat CONFIDENTIAL — selalu sebut 'Bank XYZ', JANGAN sebut nama bank asli
2. Jawab dalam Bahasa Indonesia yang profesional dan mudah dipahami manajer bank
3. Selalu sertakan angka/data spesifik dari konteks yang diberikan
4. Fokus pada insight yang ACTIONABLE — apa yang harus dilakukan, bukan hanya apa yang terjadi
5. Maksimal 3 paragraf per jawaban, kecuali diminta narasi panjang
6. Gunakan bullet point untuk rekomendasi bila perlu

FORMAT JAWABAN:
- Mulai dengan insight utama (1 kalimat)
- Elaborasi dengan data pendukung
- Akhiri dengan rekomendasi konkret


## 3. Test: Executive Summary

In [6]:
# ── Fungsi call AI (Groq) ──────────────────────────────────
def call_ai(prompt, system=SYSTEM_PROMPT, max_tokens=800):
    """Call Groq API. Gratis dan cepat."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.3,
    )
    return response.choices[0].message.content

# ── Test Executive Summary ────────────────────────────────────
exec_prompt = (
    CONTEXT
    + "\n\nTugas: Buatkan EXECUTIVE SUMMARY singkat (max 3 paragraf) dari data survei ini"
    + "\nuntuk dibaca oleh Direktur Bank XYZ. Highlight: performa NPS, cabang terbaik/terburuk,"
    + "\ndan 2 rekomendasi prioritas."
)

if API_KEY:
    result = call_ai(exec_prompt)
    print('=== EXECUTIVE SUMMARY ===')
    print(result)
else:
    print('Skip — API key belum tersedia')

=== EXECUTIVE SUMMARY ===
Insight utama dari survei kepuasan nasabah Bank XYZ menunjukkan bahwa bank ini memiliki performa NPS yang sangat baik, yaitu 80,9, dengan 82,3% responden sebagai Promoter dan hanya 1,4% sebagai Detractor. Hal ini menunjukkan bahwa mayoritas nasabah sangat puas dengan layanan Bank XYZ.

Dari data yang diberikan, terdapat beberapa cabang yang memiliki performa NPS yang sangat baik, yaitu Denpasar 2 (Bali), Lebak 1 (Banten), dan Pandeglang 3 (Banten) dengan NPS 100,0. Namun, ada juga beberapa cabang yang perlu perhatian khusus, yaitu Bogor 1 (Jawa Barat) dengan NPS -22,2, Semarang 2 (Jawa Tengah) dengan NPS 0,0, dan Tangerang 3 (Banten) dengan NPS 11,1. Berdasarkan data ini, terdapat dua rekomendasi prioritas yang dapat dilakukan oleh Bank XYZ, yaitu:
* Meningkatkan kualitas layanan di cabang-cabang yang memiliki performa NPS rendah, seperti Bogor 1 dan Semarang 2, dengan melakukan pelatihan karyawan dan memperbaiki proses operasional.
* Mengembangkan fasilitas p

## 4. Test: Q&A Analyst

In [7]:
# ── Test berbagai pertanyaan ──────────────────────────────────
test_questions = [
    'Apa insight terpenting dari hasil survei ini?',
    'Touchpoint apa yang paling perlu diperbaiki dan mengapa?',
    'Bagaimana posisi Bank XYZ dibandingkan kompetitor?',
    'Cabang mana yang harus menjadi prioritas intervensi?',
]

if API_KEY:
    for q in test_questions[:2]:  # Test 2 pertanyaan dulu
        print(f'\n=== PERTANYAAN: {q} ===')
        prompt = f'{CONTEXT}\n\nPertanyaan: {q}'
        answer = call_ai(prompt)
        print(answer)
        print('-' * 60)
else:
    print('Skip — API key belum tersedia')


=== PERTANYAAN: Apa insight terpenting dari hasil survei ini? ===
Insight utama dari hasil survei kepuasan nasabah Bank XYZ adalah bahwa nasabah secara keseluruhan sangat puas dengan layanan bank, terlihat dari NPS Score yang tinggi yaitu 80,9 dan CSI Mean yang mencapai 5,89/6, namun ada beberapa cabang yang perlu perhatian khusus karena memiliki NPS yang rendah.

Data survei menunjukkan bahwa 82,3% responden adalah promoter, yaitu nasabah yang sangat puas dan kemungkinan besar akan merekomendasikan Bank XYZ kepada orang lain. Sementara itu, hanya 1,4% responden yang merupakan detractor, yaitu nasabah yang tidak puas. Selain itu, Bank XYZ juga memiliki keunggulan brand dibandingkan dengan kompetitor, terutama dalam hal percaya diri bertransaksi, banyaknya ATM, dan membuat nasabah merasa dihargai. Driver NPS tertinggi juga menunjukkan bahwa parkir, operasional, dan banking hall merupakan faktor-faktor yang paling berpengaruh terhadap kepuasan nasabah.

Rekomendasi konkret yang dapat di

## 5. Test: Auto Narrative untuk Presentasi

In [8]:
# ── Auto Narrative untuk Presentasi ──────────────────────────
narrative_prompt = f"""{CONTEXT}

Tugas: Buatkan NARASI PRESENTASI lengkap (5-7 paragraf) dari hasil survei kepuasan nasabah
Bank XYZ. Narasi ini akan dibacakan dalam presentasi kepada manajemen senior.

Struktur narasi:
1. Pembuka — gambaran umum survei dan NPS
2. Analisis cabang — siapa terbaik dan terburuk, mengapa
3. Touchpoint — apa yang bekerja baik dan apa Quick Win prioritas
4. Emosi nasabah — perasaan dominan dan apa artinya
5. Posisi vs kompetitor
6. Rekomendasi strategis
7. Penutup — call to action"""

if API_KEY:
    narrative = call_ai(narrative_prompt, max_tokens=1200)
    print('=== AUTO NARRATIVE ===')
    print(narrative)
    with open(f'{DATA_DIR}/auto_narrative.txt', 'w', encoding='utf-8') as f:
        f.write(narrative)
    print('\n✓ Disimpan ke data/auto_narrative.txt')
else:
    print('Skip — API key belum tersedia')

=== AUTO NARRATIVE ===
Pembuka:
Hasil survei kepuasan nasabah Bank XYZ menunjukkan gambaran yang sangat positif, dengan Net Promoter Score (NPS) sebesar 80,9. Ini menandakan bahwa sebagian besar nasabah sangat puas dengan layanan yang diberikan oleh Bank XYZ. Dengan total responden sebanyak 1.730 dari 14 provinsi dan 128 cabang, survei ini memberikan wawasan yang komprehensif tentang kepuasan nasabah dan area yang perlu ditingkatkan.

Analisis cabang:
Dalam analisis cabang, terdapat beberapa cabang yang menonjol dengan NPS yang sangat tinggi, yaitu Denpasar 2 (Bali), Lebak 1 (Banten), dan Pandeglang 3 (Banten) dengan NPS 100,0. Ini menunjukkan bahwa cabang-cabang tersebut telah berhasil menyediakan layanan yang sangat memuaskan bagi nasabah. Namun, ada juga beberapa cabang yang perlu perhatian, seperti Bogor 1 (Jawa Barat) dengan NPS -22,2, Semarang 2 (Jawa Tengah) dengan NPS 0,0, dan Tangerang 3 (Banten) dengan NPS 11,1. Ini menandakan bahwa cabang-cabang tersebut perlu melakukan perb

## 6. Finalisasi Config untuk Dashboard

In [9]:
# ── Export AI config untuk dashboard ─────────────────────────
ai_config = {
    'provider': 'groq',
    'model': MODEL,
    'max_tokens': 800,
    'system_prompt': SYSTEM_PROMPT,
    'context_template': CONTEXT,
    'suggested_questions': [
        'Apa insight terpenting dari data ini?',
        'Cabang mana yang perlu prioritas perbaikan?',
        'Apa kelebihan Bank XYZ vs kompetitor?',
        'Buatkan ringkasan eksekutif untuk direksi',
        'Touchpoint apa yang paling berpengaruh ke NPS?',
        'Bagaimana profil emosi nasabah Bank XYZ?',
        'Segmen nasabah mana yang paling loyal?',
    ],
    'privacy_note': 'Data confidential — nama bank dianonimkan sebagai Bank XYZ',
}

with open(f'{DATA_DIR}/ai_config.json', 'w', encoding='utf-8') as f:
    json.dump(ai_config, f, ensure_ascii=False, indent=2)

print('✓ AI config disimpan ke data/ai_config.json')
print(f'  Provider: Groq')
print(f'  Model: {ai_config["model"]}')
print(f'  Suggested questions: {len(ai_config["suggested_questions"])}')
print('\n=== AI Analyst setup selesai! ===')
print('Selanjutnya: streamlit run dashboard.py')

✓ AI config disimpan ke data/ai_config.json
  Provider: Groq
  Model: llama-3.3-70b-versatile
  Suggested questions: 7

=== AI Analyst setup selesai! ===
Selanjutnya: streamlit run dashboard.py
